# 🤖 AI Functions Showcase - The Art of the Possible

**COMPREHENSIVE AI DEMONSTRATION** - Run this after notebook 01.

## What this notebook demonstrates:
- ✅ **ai_classify** - Intelligent priority and category classification
- ✅ **ai_extract** - Structured data extraction from unstructured text  
- ✅ **ai_gen** - Complex analysis, summaries, and creative content generation
- ✅ **Final Summary Table** - Complete AI-powered ticket analysis

**Prerequisites:** Run `01_sample_data_generation.ipynb` first

## AI Functions Showcase:
- `ai_classify` - For priority and category classification
- `ai_extract` - For structured data extraction  
- `ai_gen` - For complex analysis and summaries

---

## 🎯 Goal: Demonstrate the full power of Databricks AI Functions


In [1]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from datetime import datetime

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")
print("🚀 Ready to showcase Databricks AI Functions!")


✅ Libraries imported and configuration loaded
🎯 Using Unity Catalog: quickstart_catalog_vkm_external.classify_tickets
🚀 Ready to showcase Databricks AI Functions!


In [2]:
# Load Sample Data
print("📊 Loading sample ticket data from Unity Catalog...")

# Load the sample ticket data from Unity Catalog
df_tickets = spark.table(TABLES["raw_tickets"])
print(f"✅ Loaded {df_tickets.count()} tickets from: {TABLES['raw_tickets']}")

# Display sample data
print("\n📋 Sample ticket data:")
display(df_tickets.select("ticket_id", "short_description", "description").limit(3))


📊 Loading sample ticket data from Unity Catalog...


✅ Loaded 10 tickets from: quickstart_catalog_vkm_external.classify_tickets.raw_tickets

📋 Sample ticket data:


,ticket_id,short_description,description
0,TICKET_001,Issue #1 - Critical,hey so our website is super slow today and customers are complaining. i think it might be the database or something. can someone look into this? it's been happening since this morning and we're losing sales.
1,TICKET_002,Issue #2 - Important,urgent! the login system is broken again. users can't get in and they're calling support nonstop. this happened last week too. we need to fix this asap before more customers leave.
2,TICKET_003,Issue #3 - Problem,i need help with the new feature we're building. the api is returning weird errors and i don't know why. it works sometimes but then fails randomly. can someone debug this?


# 🎯 AI Function #1: ai_classify

**Purpose:** Intelligent classification of text into predefined categories

**Use Cases:** Priority classification, category assignment, sentiment analysis, risk assessment

**Example:** Classify ticket priorities based on description content


In [3]:
# AI Function #1: ai_classify - Priority Classification
print("🎯 Demonstrating ai_classify for priority classification...")

# Register DataFrame as temporary view for SQL access
df_tickets.createOrReplaceTempView("tickets")

# Use ai_classify to classify ticket priorities
df_with_priority = spark.sql("""
    SELECT
        *,
        ai_classify(
            description, 
            ARRAY('Low Priority', 'Medium Priority', 'High Priority', 'Urgent Priority')
        ) as ai_priority_classification
    FROM tickets
""")

print("✅ ai_classify completed - Priority classification done!")
print("\n📊 Priority Classification Results:")
display(df_with_priority.select("ticket_id", "short_description", "ai_priority_classification").limit(5))

# Show distribution of AI classifications
print("\n📈 Priority Distribution:")
df_with_priority.groupBy("ai_priority_classification").count().orderBy(desc("count")).show()


🎯 Demonstrating ai_classify for priority classification...
✅ ai_classify completed - Priority classification done!

📊 Priority Classification Results:


,ticket_id,short_description,ai_priority_classification
0,TICKET_001,Issue #1 - Critical,Urgent Priority
1,TICKET_002,Issue #2 - Important,Urgent Priority
2,TICKET_003,Issue #3 - Problem,Medium Priority
3,TICKET_004,Issue #4 - Critical,High Priority
4,TICKET_005,Issue #5 - Problem,High Priority



📈 Priority Distribution:


+--------------------------+-----+
|ai_priority_classification|count|
+--------------------------+-----+
|           Urgent Priority|    6|
|             High Priority|    3|
|           Medium Priority|    1|
+--------------------------+-----+



# 🎯 AI Function #2: ai_extract

**Purpose:** Extract structured data from unstructured text using schema definition

**Use Cases:** Action items extraction, contact information parsing, date/time extraction, entity recognition

**Example:** Extract specific action items and requirements from ticket descriptions


In [ ]:
# AI Function #2: ai_extract - Action Items Extraction
print("🎯 Demonstrating ai_extract for structured data extraction...")

# Register DataFrame as temporary view for SQL access
df_with_priority.createOrReplaceTempView("tickets_with_priority")

# Use ai_extract with proper ARRAY syntax for schema
df_with_extraction = spark.sql("""
    SELECT
        *,
        ai_extract(
            description,
            ARRAY(
                STRUCT('action_items' AS key, 'array<string>' AS value),
                STRUCT('main_requirement' AS key, 'string' AS value),
                STRUCT('urgency_level' AS key, 'string' AS value),
                STRUCT('affected_systems' AS key, 'array<string>' AS value)
            )
        ) as ai_extracted_data
    FROM tickets_with_priority
""")

print("✅ ai_extract completed - Structured data extraction done!")
print("\n📊 Extraction Results:")
display(df_with_extraction.select("ticket_id", "short_description", "ai_extracted_data").limit(3))

# Parse the extracted data for better display
df_parsed = df_with_extraction.withColumn(
    "action_items", 
    col("ai_extracted_data.action_items")
).withColumn(
    "main_requirement", 
    col("ai_extracted_data.main_requirement")
).withColumn(
    "urgency_level", 
    col("ai_extracted_data.urgency_level")
).withColumn(
    "affected_systems", 
    col("ai_extracted_data.affected_systems")
)

print("\n📋 Parsed Extraction Results:")
display(df_parsed.select("ticket_id", "action_items", "main_requirement", "urgency_level").limit(3))


🎯 Demonstrating ai_extract for structured data extraction...


{"ts": "2025-09-19 20:53:34.481", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"ai_extract(description, {\"action_items\": \"array<string>\", \"main_requirement\": \"string\", \"urgency_level\": \"string\", \"affected_systems\": \"array<string>\"})\" due to data type mismatch: The second parameter requires the \"ARRAY\" type, however \"{\"action_items\": \"array<string>\", \"main_requirement\": \"string\", \"urgency_level\": \"string\", \"affected_systems\": \"array<string>\"}\" has the type \"STRING\". SQLSTATE: 42K09; line 4 pos 8;\n'Project [assigned_to#114162, assignment_group#114163, description#114164, priority#114165, requested_by#114166, short_description#114167, state#114168, ticket_id#114169, ai_priority_classification#114202, ai_extract(description#114164, {\"action_items\": \"array<string>\", \"main_requirement\": \"string\", \"urgency_level\": \"string\", \"affected_systems\": \"array<string>\"}) AS 

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "ai_extract(description, {"action_items": "array<string>", "main_requirement": "string", "urgency_level": "string", "affected_systems": "array<string>"})" due to data type mismatch: The second parameter requires the "ARRAY" type, however "{"action_items": "array<string>", "main_requirement": "string", "urgency_level": "string", "affected_systems": "array<string>"}" has the type "STRING". SQLSTATE: 42K09; line 4 pos 8;
'Project [assigned_to#114162, assignment_group#114163, description#114164, priority#114165, requested_by#114166, short_description#114167, state#114168, ticket_id#114169, ai_priority_classification#114202, ai_extract(description#114164, {"action_items": "array<string>", "main_requirement": "string", "urgency_level": "string", "affected_systems": "array<string>"}) AS ai_extracted_data#114246]
+- SubqueryAlias tickets_with_priority
   +- View (`tickets_with_priority`, [assigned_to#114162, assignment

# 🎯 AI Function #3: ai_gen

**Purpose:** Generate creative content, summaries, and complex analysis

**Use Cases:** Executive summaries, detailed analysis, creative content, recommendations

**Example:** Generate comprehensive ticket analysis and recommendations


In [ ]:
# AI Function #3: ai_gen - Comprehensive Analysis
print("🎯 Demonstrating ai_gen for comprehensive analysis...")

# Register DataFrame as temporary view for SQL access
df_parsed.createOrReplaceTempView("tickets_with_extraction")

# Use ai_gen to create comprehensive analysis
df_with_analysis = spark.sql("""
    SELECT
        *,
        ai_gen(
            CONCAT(
                'Analyze this IT ticket and provide: ',
                '1. Executive summary (2-3 sentences), ',
                '2. Technical complexity (Low/Medium/High), ',
                '3. Estimated effort (hours), ',
                '4. Risk assessment (Low/Medium/High), ',
                '5. Recommended next steps. ',
                'Ticket: ', short_description, ' - ', description
            )
        ) as ai_comprehensive_analysis
    FROM tickets_with_extraction
""")

print("✅ ai_gen completed - Comprehensive analysis done!")
print("\n📊 Analysis Results:")
display(df_with_analysis.select("ticket_id", "short_description", "ai_comprehensive_analysis").limit(3))


# 🎉 Final Summary Table - The Art of the Possible

**Complete AI-powered ticket analysis showcasing all three AI functions**


In [ ]:
# Create Final Summary Table
print("🎉 Creating comprehensive summary table...")

# Create a clean summary table with all AI results
df_final_summary = df_with_analysis.select(
    "ticket_id",
    "short_description",
    "ai_priority_classification",
    "action_items",
    "main_requirement", 
    "urgency_level",
    "affected_systems",
    "ai_comprehensive_analysis"
).withColumn(
    "ai_showcase_timestamp", 
    current_timestamp()
)

print("✅ Final summary table created!")
print(f"📊 Total tickets analyzed: {df_final_summary.count()}")

# Display the comprehensive results
print("\n🎯 COMPLETE AI SHOWCASE RESULTS:")
print("="*80)
display(df_final_summary.limit(5))

# Save to Unity Catalog
print("\n💾 Saving results to Unity Catalog...")
df_final_summary.write.format("delta").mode("overwrite").saveAsTable(TABLES["ai_showcase_results"])
print(f"✅ Results saved to: {TABLES['ai_showcase_results']}")

# Show summary statistics
print("\n📈 AI Showcase Summary Statistics:")
print(f"🎯 Tickets processed: {df_final_summary.count()}")
print(f"🤖 AI functions demonstrated: 3 (ai_classify, ai_extract, ai_gen)")
print(f"📊 Data saved to Unity Catalog: {TABLES['ai_showcase_results']}")

print("\n" + "="*80)
print("🎉 AI SHOWCASE COMPLETED SUCCESSFULLY!")
print("="*80)
print("✅ ai_classify: Priority classification")
print("✅ ai_extract: Structured data extraction") 
print("✅ ai_gen: Comprehensive analysis")
print("✅ Final summary table created and saved")
print("="*80)
